In [ ]:
import os

PROJECT_NAME = "adam_and_eve"

# Text to mine for stylistic examples
# file path or URL (e.g. https://www.gutenberg.org/files/84/84-0.txt)
TEXT_SOURCE = ""

# Describe the style you're looking for — the agent uses this to decide what's worth keeping
STYLE_DESCRIPTION = """
Introspective science fiction with poetic, sparse prose. First-person voice.
Passages that blend the philosophical with the visceral. Short, punchy paragraphs.
""".strip()

# Target number of passages to extract from the text
N_PASSAGES = 15

# Words per chunk sent to the model (keep under ~4000 to avoid slow responses)
CHUNK_SIZE_WORDS = 3000

# Minimum quality score (1-10) to include a passage as a candidate for review
MIN_QUALITY = 6

# Scanning model: fast + cheap — haiku is fine here
SCAN_MODEL = "claude-haiku-4-5-20251001"

OUTPUT_FILE = f"finetuning_data/{PROJECT_NAME}/source_sections.jsonl"

In [ ]:
import requests

def load_text(source: str) -> str:
    if source.startswith("http"):
        r = requests.get(source, timeout=30)
        r.raise_for_status()
        return r.text
    with open(source, encoding="utf-8", errors="replace") as f:
        return f.read()

if not TEXT_SOURCE:
    raise ValueError("Set TEXT_SOURCE to a file path or URL")

raw_text = load_text(TEXT_SOURCE)

# Split into words then rechunk to avoid cutting mid-sentence
words = raw_text.split()
chunks = []
for i in range(0, len(words), CHUNK_SIZE_WORDS):
    chunk = " ".join(words[i : i + CHUNK_SIZE_WORDS])
    chunks.append(chunk)

print(f"Text: {len(words):,} words → {len(chunks)} chunks of ~{CHUNK_SIZE_WORDS} words each")

In [ ]:
import anthropic
import json
from tqdm.notebook import tqdm

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

SCAN_PROMPT = """\
You are a literary editor mining a text for stylistically useful passages.

Target style:
{style}

From the excerpt below, extract 0–3 passages that exemplify the target style.
Each passage must be:
- Copied verbatim from the text (exact words, no paraphrasing)
- Self-contained (a complete paragraph or short run of 2-3 consecutive paragraphs)
- Between 50 and 400 words

Return ONLY valid JSON, nothing else:
{{"passages": [
  {{"text": "<verbatim passage>", "reason": "<one sentence why>", "score": <1-10>}}
]}}

If no passage in this excerpt meets the bar, return {{"passages": []}}.

Excerpt:
---
{chunk}
---"""


def scan_chunk(chunk: str) -> list[dict]:
    prompt = SCAN_PROMPT.format(style=STYLE_DESCRIPTION, chunk=chunk)
    response = client.messages.create(
        model=SCAN_MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()
    # Strip markdown code fences if present
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    try:
        data = json.loads(text)
        return data.get("passages", [])
    except json.JSONDecodeError:
        print(f"  Warning: failed to parse JSON response: {text[:200]}")
        return []


# Scan all chunks
all_candidates = []
for i, chunk in enumerate(tqdm(chunks, desc="Scanning")):
    passages = scan_chunk(chunk)
    for p in passages:
        p["chunk_idx"] = i
    all_candidates.extend(passages)

print(f"\nFound {len(all_candidates)} raw candidates across {len(chunks)} chunks")

In [ ]:
# Filter by quality score and deduplicate near-identical passages
filtered = [c for c in all_candidates if c.get("score", 0) >= MIN_QUALITY]

# Simple dedup: drop candidates whose first 80 chars match an already-kept one
seen_prefixes = set()
deduped = []
for c in sorted(filtered, key=lambda x: -x.get("score", 0)):
    prefix = c["text"][:80].strip()
    if prefix not in seen_prefixes:
        seen_prefixes.add(prefix)
        deduped.append(c)

# Take top N by score
candidates = deduped[:N_PASSAGES]

print(f"After filtering (score ≥ {MIN_QUALITY}) and dedup: {len(deduped)} candidates")
print(f"Presenting top {len(candidates)} for review")

# Score distribution
from collections import Counter
scores = Counter(c.get("score") for c in all_candidates)
for score in sorted(scores, reverse=True):
    print(f"  Score {score}: {scores[score]} passages")

In [ ]:
# Quick review: approve or skip each candidate
# The agent did the searching — you just decide keep/skip

import ipywidgets as widgets
from IPython.display import display, clear_output


class ReviewUI:
    def __init__(self, candidates, output_file):
        self.candidates = candidates
        self.output_file = output_file
        self.idx = 0
        self.approved = []

        self.progress = widgets.HTML()
        self.score_label = widgets.HTML()
        self.reason_label = widgets.HTML()
        self.passage_display = widgets.HTML(
            layout=widgets.Layout(
                border="1px solid #dee2e6", padding="16px",
                min_height="200px", max_height="500px",
                overflow_y="scroll", width="100%",
            )
        )
        self.keep_btn = widgets.Button(
            description="Keep ✓", button_style="success",
            layout=widgets.Layout(width="120px", height="40px"),
        )
        self.skip_btn = widgets.Button(
            description="Skip →", button_style="",
            layout=widgets.Layout(width="120px", height="40px"),
        )
        self.save_btn = widgets.Button(
            description="Save All", button_style="warning",
            layout=widgets.Layout(width="120px", height="40px"),
        )
        self.msg = widgets.Output()

        self.keep_btn.on_click(self._keep)
        self.skip_btn.on_click(self._skip)
        self.save_btn.on_click(self._save)

        self._render()

    def _render(self):
        if self.idx >= len(self.candidates):
            self.passage_display.value = '<p style="color:#6c757d">All candidates reviewed.</p>'
            self.keep_btn.disabled = True
            self.skip_btn.disabled = True
            self.progress.value = f'<b>Done — {len(self.approved)} kept of {len(self.candidates)}</b>'
            return

        c = self.candidates[self.idx]
        n = len(self.candidates)
        n_left = n - self.idx
        self.progress.value = (
            f'<span style="color:#6c757d">{self.idx + 1} / {n} &nbsp;·&nbsp; '
            f'{len(self.approved)} kept &nbsp;·&nbsp; {n_left} remaining</span>'
        )
        score = c.get("score", "?")
        color = "#28a745" if score >= 8 else ("#ffc107" if score >= 6 else "#dc3545")
        self.score_label.value = f'<b style="color:{color}">Score: {score}/10</b>'
        self.reason_label.value = (
            f'<span style="font-style:italic; color:#495057">{c.get("reason", "")}'
            f'</span>'
        )
        text_html = c["text"].replace("\n\n", "<br><br>").replace("\n", "<br>")
        self.passage_display.value = (
            f'<div style="font-family:Georgia,serif; font-size:14px; line-height:1.7;">{text_html}</div>'
        )

    def _keep(self, _):
        if self.idx < len(self.candidates):
            self.approved.append(self.candidates[self.idx])
        self.idx += 1
        self._render()

    def _skip(self, _):
        self.idx += 1
        self._render()

    def _save(self, _):
        if not self.approved:
            with self.msg:
                clear_output(); print("Nothing approved yet.")
            return
        os.makedirs(os.path.dirname(self.output_file) or ".", exist_ok=True)
        with open(self.output_file, "w") as f:
            for entry in self.approved:
                f.write(json.dumps({
                    "type": "chosen",
                    "text": entry["text"],
                    "source": TEXT_SOURCE,
                    "reason": entry.get("reason", ""),
                    "score": entry.get("score"),
                }) + "\n")
        with self.msg:
            clear_output()
            print(f"Saved {len(self.approved)} passages → {self.output_file}")

    def show(self):
        display(widgets.VBox([
            widgets.HTML('<h3 style="margin-bottom:4px">Review Candidates</h3>'),
            self.progress,
            widgets.HTML('<hr style="margin:6px 0">'),
            self.score_label,
            self.reason_label,
            self.passage_display,
            widgets.HBox([self.keep_btn, self.skip_btn, self.save_btn]),
            self.msg,
        ]))


if not candidates:
    print("No candidates found — try lowering MIN_QUALITY or changing STYLE_DESCRIPTION")
else:
    reviewer = ReviewUI(candidates, OUTPUT_FILE)
    reviewer.show()

In [ ]:
# Summary of what was saved
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        saved = [json.loads(line) for line in f if line.strip()]
    print(f"{len(saved)} passages in {OUTPUT_FILE}")
    for i, s in enumerate(saved):
        wc = len(s["text"].split())
        print(f"  [{i+1}] score={s.get('score','?')} {wc}w — {s.get('reason','')[:80]}")
else:
    print("Nothing saved yet — click 'Save All' in the review UI above")